# 🖋️ Aksara Jawa Classifier Demo
**School Project — Image Preprocessing & Classic ML**

This notebook classifies a handwritten Aksara Jawa character using:
1. **Otsu Thresholding** — Auto-binarizes the image (hand-written with NumPy)
2. **Morphological Closing** — Fills gaps in handwriting strokes (hand-written loops)
3. **Character Centering** — Crops and centers the ink so position doesn't confuse the model
4. **Random Forest Classifier** — Classic ML trained via `aksara_train_v3.py`

In [ ]:
# ── Install & Import ──────────────────────────────────────────────────────────
!pip install scikit-learn Pillow numpy matplotlib joblib -q

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import joblib
from google.colab import files

## Step 1: Upload Your Trained Model
Run the cell below and upload `aksara_jawa_v3_model.joblib` from your computer.

In [ ]:
# Upload the saved model from your local machine
print("Please upload: aksara_jawa_v3_model.joblib")
uploaded = files.upload()

# Load the model and class list
model_data = joblib.load('aksara_jawa_v3_model.joblib')
rf_model  = model_data['model']
classes   = model_data['classes']
IMG_SIZE  = model_data['img_size']  # (64, 64)

print(f"Model loaded! Classes: {classes}")
print(f"Expected image input size: {IMG_SIZE}")

---
## Step 2: Preprocessing Functions
These are the **exact same** functions used in `aksara_train_v3.py` — copied here verbatim.

In [ ]:
def otsu_threshold_verbose(gray: np.ndarray) -> int:
    hist, _ = np.histogram(gray.flatten(), bins=256, range=(0, 256))
    total_pixels = gray.size
    best_threshold = 0
    max_variance   = 0
    for t in range(256):
        weight_bg = np.sum(hist[:t]) / total_pixels
        weight_fg = np.sum(hist[t:]) / total_pixels
        if weight_bg == 0 or weight_fg == 0: continue
        mean_bg = np.sum(np.arange(t) * hist[:t]) / (np.sum(hist[:t]) + 1e-10)
        mean_fg = np.sum(np.arange(t, 256) * hist[t:]) / (np.sum(hist[t:]) + 1e-10)
        variance = weight_bg * weight_fg * (mean_bg - mean_fg) ** 2
        if variance > max_variance:
            max_variance = variance
            best_threshold = t
    return best_threshold

def apply_threshold(gray: np.ndarray, t: int) -> np.ndarray:
    return np.where(gray <= t, 255, 0).astype(np.uint8)

def erode_verbose(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    H, W = image.shape
    pad = kernel_size // 2
    output = np.zeros_like(image)
    padded = np.pad(image, pad_width=pad, mode='constant', constant_values=0)
    for i in range(H):
        for j in range(W):
            current_min = 255
            for ri in range(kernel_size):
                for ci in range(kernel_size):
                    if padded[i + ri, j + ci] < current_min: current_min = padded[i + ri, j + ci]
            output[i, j] = current_min
    return output

def dilate_verbose(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    H, W = image.shape
    pad = kernel_size // 2
    output = np.zeros_like(image)
    padded = np.pad(image, pad_width=pad, mode='constant', constant_values=0)
    for i in range(H):
        for j in range(W):
            current_max = 0
            for ri in range(kernel_size):
                for ci in range(kernel_size):
                    if padded[i + ri, j + ci] > current_max: current_max = padded[i + ri, j + ci]
            output[i, j] = current_max
    return output

def closing_verbose(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    return erode_verbose(dilate_verbose(image, kernel_size), kernel_size)

def center_character(binary: np.ndarray, target_size=(64, 64)) -> np.ndarray:
    coords = np.argwhere(binary == 255)
    if coords.size == 0: return np.zeros(target_size, dtype=np.uint8)
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    crop = binary[y_min:y_max+1, x_min:x_max+1]
    new_img = np.zeros(target_size, dtype=np.uint8)
    ch, cw = crop.shape
    th, tw = target_size
    if ch > th or cw > tw:
        crop = np.array(Image.fromarray(crop).resize((tw-4, th-4), Image.Resampling.LANCZOS))
        ch, cw = crop.shape
    y_off, x_off = (th-ch)//2, (tw-cw)//2
    new_img[y_off:y_off+ch, x_off:x_off+cw] = crop
    return new_img

## Step 3: Run Pipeline and Classify
Upload your image and see the results.

In [ ]:
print("Upload an image file:")
up = files.upload()
fname = list(up.keys())[0]

img = Image.open(fname).convert('L')
gray = np.array(img)
t = otsu_threshold_verbose(gray)
binary = apply_threshold(gray, t)
cleaned = closing_verbose(binary, kernel_size=3)
centered = center_character(cleaned, target_size=IMG_SIZE)

feat = centered.flatten().reshape(1, -1)
pred_idx = rf_model.predict(feat)[0]
probs = rf_model.predict_proba(feat)[0]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title("Original")
plt.subplot(1, 2, 2); plt.imshow(centered, cmap='gray'); plt.title(f"Prediction: {classes[pred_idx]}")
plt.show()

print(f"Result: {classes[pred_idx]} ({probs[pred_idx]*100:.1f}% confidence)")